# Evaluation Report

This is an ablation report evaluating the performance of 3 components / component combinations of a Sequential Recommender System. The models evaluated are:

| Model | Description |
|-------|-------------|
| **Retriever A** | Item2Vec + FAISS index |
| **Ranker A** | LACLRec trained on all data |
| **Combined B** | Retriever A (generate candidates) → Ranker A|

---
## Train

In [2]:
import sys, os, subprocess
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent

os.environ['RATINGS_CSV']  = str(PROJECT_ROOT / 'datasets/ml-latest-small/ratings.csv')
os.environ['PROCESSED_PT'] = str(PROJECT_ROOT / 'datasets/processed.pt')
os.environ['CKPT_DIR']     = str(PROJECT_ROOT / 'checkpoints')

def run(cmd):
    result = subprocess.run(cmd, shell=True, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

run('python train.py --mode process')
run('python train.py --mode retriever_a')
run('python train.py --mode ranker_a')

[MLDataset] saved to /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/datasets/processed.pt

[train] loading processed dataset from /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/datasets/processed.pt
[MLDataset] loaded from /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/datasets/processed.pt  (2299 samples)
[train] --- Retriever A (Item2Vec, all data) ---
[train] retriever_a saved → /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/checkpoints/retriever_a.pkl
[Item2Vec] FAISS index saved → /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/checkpoints/retriever_a.faiss  (9701 items)

[train] loading processed dataset from /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/datasets/processed.pt
[MLDataset] loaded from /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems

In [3]:
import torch
import pandas as pd
from torch.utils.data import DataLoader

from src.data.dataset import MLDataset
from src.retrieval.Item2Vec import Item2VecRetriever
from src.models.ranker import build_ranker
from src.models.retriever_ranker import load_combined
from src.evaluation.evaluate import (
    evaluate_Item2Vec,
    evaluate_LACLRec,
    evaluate_RetrieverRanker,
)

In [4]:
CKPT_DIR   = str(PROJECT_ROOT / 'checkpoints')
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN    = 50
EMBED_DIM  = 64
INSERT_LEN = 3
K          = 10

dataset = MLDataset.load_processed(str(PROJECT_ROOT / 'datasets/processed.pt'))
loader  = DataLoader(dataset, batch_size=64, shuffle=False)
print(f'{len(dataset):,} samples | {dataset.num_items:,} items | device: {DEVICE}')

[MLDataset] loaded from /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/datasets/processed.pt  (2299 samples)
2,299 samples | 9,725 items | device: cpu


---
## Retriever A — Item2Vec + FAISS

In [5]:
retriever_a = Item2VecRetriever.load(f'{CKPT_DIR}/retriever_a.pkl')
retriever_a.load_faiss_index(f'{CKPT_DIR}/retriever_a.faiss')

metrics_retriever_a = evaluate_Item2Vec(retriever_a, loader, k=K)
print('Retriever A:', metrics_retriever_a)

[Item2Vec] FAISS index loaded ← /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/checkpoints/retriever_a.faiss  (9701 items)
Retriever A: {'Recall@10': 0.013888888888888888, 'NDCG@10': 0.006440871578708738}


---
## Ranker A — LACLRec

In [6]:
enc_a, _, rec_a = build_ranker(
    n_items=dataset.num_items,
    seq_len=SEQ_LEN,
    embed_dim=EMBED_DIM,
    insert_len=INSERT_LEN,
)
enc_a.load_state_dict(torch.load(f'{CKPT_DIR}/ranker_a_encoder.pt',     map_location=DEVICE))
rec_a.load_state_dict(torch.load(f'{CKPT_DIR}/ranker_a_recommender.pt', map_location=DEVICE))
enc_a.to(DEVICE).eval()
rec_a.to(DEVICE).eval()

metrics_ranker_a = evaluate_LACLRec(rec_a, loader, device=DEVICE, k=K)
print('Ranker A:', metrics_ranker_a)

Ranker A: {'Recall@10': 0.11247204575273725, 'NDCG@10': 0.05568537561975034}


---
## Combined B — Retriever A → Ranker A

In [7]:
combined_b = load_combined(
    dataset=dataset,
    ckpt_dir=CKPT_DIR,
    device=DEVICE,
    n_candidates=1000,
    seq_len=SEQ_LEN,
    embed_dim=EMBED_DIM,
    insert_len=INSERT_LEN,
)

metrics_combined_b = evaluate_RetrieverRanker(combined_b, loader, k=K)
print('Combined B:', metrics_combined_b)

[Item2Vec] FAISS index loaded ← /home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp (Copy 3)/project1/checkpoints/retriever_a.faiss  (9701 items)
Combined B: {'Recall@10': 0.07690383700860871, 'NDCG@10': 0.03595323921296433}


---
## Summary

In [9]:
summary = pd.DataFrame([
    {'Model': 'Retriever A  (Item2Vec + FAISS)',    **metrics_retriever_a},
    {'Model': 'Ranker A     (LACLRec)',             **metrics_ranker_a},
    {'Model': 'Combined B   (Retriever A + Ranker A)',  **metrics_combined_b},
]).set_index('Model')

summary.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen')

,Recall@10,NDCG@10
Model,,
Retriever A (Item2Vec + FAISS),0.0139,0.0064
Ranker A (LACLRec),0.1125,0.0557
Combined B (Retriever A + Ranker A),0.0769,0.0360
